# Demo Chương 6: Neural Networks trong NLP 


1. **Word Embeddings (Ma trận E)**: Biểu diễn từ thành các vector liên tục (Mục 6.5).
2. **Mean-Pooling**: Gộp các vector từ trong câu thành một vector duy nhất đại diện cho câu (Công thức 6.21).
3. **Feedforward Neural Network (MLP)**: Mạng nơ-ron truyền thẳng với các lớp ẩn (Hidden layers) và hàm kích hoạt phi tuyến (ReLU) (Mục 6.3).
4. **Cross-Entropy Loss & Backpropagation**: Huấn luyện mạng (Mục 6.6).

**Dataset**: IMDB Movie Reviews (Kaggle) - Phân loại đánh giá phim Tích cực (Positive) hoặc Tiêu cực (Negative).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Thiết lập device (sử dụng GPU nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cpu


## 1. Tải và Tiền xử lý Dữ liệu

In [2]:
# Đọc dữ liệu từ file CSV (Tải từ Kaggle)
# Giả sử file đã được tải về với tên 'IMDB Dataset.csv'
try:
    df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
except FileNotFoundError:
    print("Vui lòng tải file 'IMDB Dataset.csv' từ Kaggle và đặt vào thư mục hiện tại.")
    # Tạo dummy data nếu không có file để code vẫn chạy được minh họa
    df = pd.DataFrame({
        'review':['This movie is great', 'Terrible film, I hate it', 'Wonderful acting and plot', 'Bad bad bad'],
        'sentiment':['positive', 'negative', 'positive', 'negative']
    })

# Ánh xạ nhãn: positive -> 1, negative -> 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

def clean_text(text):
    text = re.sub(r'<br />', ' ', text) # Xóa thẻ HTML
    text = re.sub(r'[^\w\s]', '', text) # Xóa dấu câu
    return text.lower().split()

print("Đang tiền xử lý văn bản...")
df['tokens'] = df['review'].apply(clean_text)
print("Số lượng mẫu:", len(df))
df.head()

Đang tiền xử lý văn bản...
Số lượng mẫu: 50000


,review,sentiment,label,tokens
0,One of the other reviewers has mentioned that ...,positive,1,"[one, of, the, other, reviewers, has, mentione..."
1,A wonderful little production. <br /><br />The...,positive,1,"[a, wonderful, little, production, the, filmin..."
2,I thought this was a wonderful way to spend ti...,positive,1,"[i, thought, this, was, a, wonderful, way, to,..."
3,Basically there's a family where a little boy ...,negative,0,"[basically, theres, a, family, where, a, littl..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,"[petter, matteis, love, in, the, time, of, mon..."


In [3]:
# Xây dựng từ điển (Vocabulary)
vocab = Counter()
for tokens in df['tokens']:
    vocab.update(tokens)

# Chỉ giữ lại các từ xuất hiện ít nhất 5 lần
min_freq = 5
word2idx = {'<PAD>': 0, '<UNK>': 1} # Padding và Unknown tokens
idx = 2
for word, count in vocab.items():
    if count >= min_freq:
        word2idx[word] = idx
        idx += 1

vocab_size = len(word2idx)
print(f"Kích thước từ vựng (Vocabulary size |V|): {vocab_size}")

# Hàm chuyển đổi list các từ thành list các index
def encode_text(tokens):
    return[word2idx.get(word, word2idx['<UNK>']) for word in tokens]

df['encoded'] = df['tokens'].apply(encode_text)

Kích thước từ vựng (Vocabulary size |V|): 43115


## 2. Tạo PyTorch Dataset & DataLoader

In [4]:
class IMDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return torch.tensor(self.X.iloc[idx], dtype=torch.long), torch.tensor(self.y.iloc[idx], dtype=torch.float)

# Chia Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(df['encoded'], df['label'], test_size=0.2, random_state=42)

train_dataset = IMDBDataset(X_train, y_train)
test_dataset = IMDBDataset(X_test, y_test)

# Collate function để padding các câu có độ dài khác nhau trong cùng 1 batch
def collate_fn(batch):
    texts, labels = zip(*batch)
    offsets = [0] +[len(t) for t in texts[:-1]]
    offsets = torch.tensor(offsets).cumsum(dim=0)
    texts = torch.cat(texts) # Ghép nối các tokens thành 1 mảng 1D dài (dùng cho EmbeddingBag)
    labels = torch.tensor(labels)
    return texts, offsets, labels

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

## 3. Định nghĩa Mạng Neural Feedforward (Bám sát Chương 6)

In [5]:
class FeedforwardNeuralNet(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(FeedforwardNeuralNet, self).__init__()
        
        # 1. Embedding Layer kết hợp Mean-Pooling (Đại diện cho ma trận E)
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode='mean', sparse=False)
        
        # 2. Hidden Layer (W và b)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        
        # 3. Activation Function (f(z)) -> ReLU (Mục 6.1)
        self.relu = nn.ReLU()
        
        # 4. Output Layer (U)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
        # Lưu ý: Hàm Sigmoid được tích hợp trong BCEWithLogitsLoss ở phần Huấn luyện để ổn định số học
        
    def forward(self, text, offsets):
        # Tính e(w_i) và tính trung bình x_mean (Pooling)
        x_mean = self.embedding(text, offsets)  # Shape: [batch_size, embed_dim]
        
        # Tính lớp ẩn: h = ReLU(W * x_mean + b)
        z1 = self.fc1(x_mean)
        h = self.relu(z1)                       # Shape: [batch_size, hidden_dim]
        
        # Tính logits đầu ra: z = U * h
        z2 = self.fc2(h)                        # Shape:[batch_size, 1]
        
        return z2.squeeze(1) # Trả về vector 1D

# Khởi tạo mô hình
EMBED_DIM = 64
HIDDEN_DIM = 32
OUTPUT_DIM = 1 # Bài toán Binary Classification

model = FeedforwardNeuralNet(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
print(model)

FeedforwardNeuralNet(
  (embedding): EmbeddingBag(43115, 64, mode='mean')
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


## 4. Huấn luyện Mô hình 

In [6]:
criterion = nn.BCEWithLogitsLoss() # Kết hợp Sigmoid và Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

def binary_accuracy(preds, y):
    # Áp dụng sigmoid và làm tròn để lấy nhãn dự đoán (0 hoặc 1)
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    epoch_acc = 0
    
    for texts, offsets, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        texts, offsets, labels = texts.to(device), offsets.to(device), labels.to(device)
        
        optimizer.zero_grad() # Xóa gradient cũ
        
        # Lan truyền xuôi (Forward pass)
        predictions = model(texts, offsets)
        
        # Tính Loss
        loss = criterion(predictions, labels)
        acc = binary_accuracy(predictions, labels)
        
        # Lan truyền ngược (Backpropagation - Mục 6.6.4)
        loss.backward()
        
        # Cập nhật trọng số (W, b)
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    print(f"Train Loss: {epoch_loss/len(train_loader):.4f} | Train Acc: {epoch_acc/len(train_loader):.4f}")

Epoch 1/5: 100%|██████████| 625/625 [00:14<00:00, 44.06it/s]


Train Loss: 0.4862 | Train Acc: 0.7652


Epoch 2/5: 100%|██████████| 625/625 [00:13<00:00, 46.43it/s]


Train Loss: 0.2807 | Train Acc: 0.8875


Epoch 3/5: 100%|██████████| 625/625 [00:13<00:00, 46.20it/s]


Train Loss: 0.2126 | Train Acc: 0.9196


Epoch 4/5: 100%|██████████| 625/625 [00:13<00:00, 45.82it/s]


Train Loss: 0.1684 | Train Acc: 0.9396


Epoch 5/5: 100%|██████████| 625/625 [00:13<00:00, 46.51it/s]

Train Loss: 0.1340 | Train Acc: 0.9546


## 5. Đánh giá trên tập Kiểm thử (Evaluation)

In [7]:
model.eval()
test_loss = 0
test_acc = 0

with torch.no_grad(): # Tắt tính toán gradient
    for texts, offsets, labels in test_loader:
        texts, offsets, labels = texts.to(device), offsets.to(device), labels.to(device)
        
        predictions = model(texts, offsets)
        loss = criterion(predictions, labels)
        acc = binary_accuracy(predictions, labels)
        
        test_loss += loss.item()
        test_acc += acc.item()

print(f"Test Loss: {test_loss/len(test_loader):.4f} | Test Acc: {test_acc/len(test_loader):.4f}")

Test Loss: 0.2670 | Test Acc: 0.8980


## 6. Inference Demo (Sử dụng model để dự đoán thực tế)

In [8]:
def predict_sentiment(model, sentence):
    model.eval()
    # Tiền xử lý giống hệt lúc train
    tokens = clean_text(sentence)
    indexed = encode_text(tokens)
    
    tensor_text = torch.tensor(indexed).to(device)
    offsets = torch.tensor([0]).to(device) # Bắt đầu từ index 0 vì chỉ có 1 câu
    
    with torch.no_grad():
        prediction = model(tensor_text, offsets)
        probability = torch.sigmoid(prediction).item()
        
    sentiment = "Tích cực (Positive)" if probability >= 0.5 else "Tiêu cực (Negative)"
    return sentiment, probability

# Chạy thử một vài câu
reviews_to_test =[
    "This movie was fantastic, the acting was brilliant and I loved the story!",
    "What a waste of time. The plot was boring and the acting was terrible.",
    "It was okay, not the best but not the worst either."
]

for review in reviews_to_test:
    sentiment, prob = predict_sentiment(model, review)
    print(f"Review: '{review}'")
    print(f"-> Dự đoán: {sentiment} (Xác suất Positive: {prob:.4f})\n")

Review: 'This movie was fantastic, the acting was brilliant and I loved the story!'
-> Dự đoán: Tích cực (Positive) (Xác suất Positive: 1.0000)

Review: 'What a waste of time. The plot was boring and the acting was terrible.'
-> Dự đoán: Tiêu cực (Negative) (Xác suất Positive: 0.0000)

Review: 'It was okay, not the best but not the worst either.'
-> Dự đoán: Tiêu cực (Negative) (Xác suất Positive: 0.0000)

